In [77]:
from scipy.io import loadmat
import numpy as np
import torch

file_path = "BCICIV_1calib_1000Hz_mat/BCICIV_calib_ds1a_1000Hz.mat"
device = torch.device('cuda')

mat = loadmat(file_path)

mat

{'__header__': b'MATLAB 5.0 MAT-file, Platform: GLNXA64, Created on: Wed Jul  2 19:31:49 2008',
 '__version__': '1.0',
 '__globals__': [],
 'cnt': array([[ -97, -100, -133, ...,  -26,   74,   28],
        [ -32,  -70,  -34, ...,   37,  136,   84],
        [   8,   -7,   19, ...,  110,  203,  162],
        ...,
        [2500, 2590, 2463, ..., 1364, 1850, 1197],
        [2422, 2528, 2338, ..., 1311, 1790, 1148],
        [2364, 2469, 2261, ..., 1248, 1714, 1106]], dtype=int16),
 'mrk': array([[(array([[  20913,   28913,   36913,   44913,   52914,   60914,   68914,
                   76915,   84915,   92915,  100915,  108916,  116916,  124916,
                  132917,  162940,  170941,  178941,  186941,  194942,  202942,
                  210942,  218942,  226943,  234943,  242943,  250944,  258944,
                  266944,  274944,  304948,  312949,  320949,  328949,  336950,
                  344950,  352950,  360950,  368951,  376951,  384951,  392952,
                  400952,  40895

In [78]:
freq = mat["nfo"]["fs"][0][0][0][0]

image_display_time_s = 4
blank_display_time_s = 2
fixation_only_display_time_s = 2


eeg_signal = mat["cnt"]
n_channels = eeg_signal.shape[1]
n_samples =  eeg_signal.shape[0]
n_classes = 3
batch_size = 256

target_classes = mat["mrk"]["y"][0][0][0]
image_show_index = mat["mrk"]["pos"][0][0][0]

In [79]:
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F

target_classes_3 = np.zeros(n_samples)

image_samples_duration = freq * image_display_time_s

for index, class_number in zip(image_show_index, target_classes):
    target_classes_3[index:index+image_samples_duration] = class_number

target_classes_onehot = F.one_hot(torch.tensor(target_classes_3 + 1).long(), num_classes=n_classes).float()

dataset = TensorDataset(torch.tensor(eeg_signal).float(), torch.tensor(target_classes_onehot).float())
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)


/tmp/ipykernel_73888/2117330959.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dataset = TensorDataset(torch.tensor(eeg_signal).float(), torch.tensor(target_classes_onehot).float())


In [90]:
print("EEG signal:")
print(eeg_signal.shape)

print()

print("One Hot encoded classes corresponding to signal")
print(target_classes_onehot.shape)

EEG signal:
(1905940, 59)

One Hot encoded classes corresponding to signal
torch.Size([1905940, 3])


In [ ]:
from torchesn.nn import ESN
import torch
import torch.nn.functional as F

input_size = n_channels      # np. 59
hidden_size = 100
output_size = n_classes
washout_rate = 0.01

model = ESN(
    input_size=input_size,
    hidden_size=hidden_size,
    output_size=output_size,
    output_steps="mean",
    readout_training='cholesky'
).to(device)

def reshape_batch(batch):
    batch = batch.view(batch.size(0), batch.size(1), -1)
    return batch.transpose(0, 1).transpose(0, 2)


total_batches = len(dataloader)

# Fit the model z ewaluacją
for i, batch in enumerate(dataloader):
    if i >= 200:
        break
    x, y = batch
    x = reshape_batch(x)
    
    washout_list = torch.tensor([int(washout_rate * x.size(0))] * x.size(1)).to(device)
    x = x.to(device)
    y = y.to(device)
    
    # forward + zebranie statystyk do readoutu
    output, _ = model(x, washout_list, None, y)
    
    # fit readoutu
    model.fit()
    
    # -----------------------
    # ewaluacja po trenowaniu readoutu
    # -----------------------
    with torch.no_grad():
        y_pred, _ = model(x, washout_list)
        y_index = y.argmax(dim=-1)
        loss = F.cross_entropy(y_pred.reshape(-1, n_classes), y_index.reshape(-1))
        
        pred_classes = y_pred.argmax(dim=-1)
        accuracy = (pred_classes == y_index).float().mean()

    percent_done = 100 * (i + 1) / total_batches
    print(f"Batch {i+1}/{total_batches} ({percent_done:.1f}%) | Loss: {loss.item():.4f} | Accuracy: {accuracy.item():.4f}")


Batch 1/7446 (0.0%) | Loss: 1.0960 | Accuracy: 1.0000
Batch 2/7446 (0.0%) | Loss: 1.0960 | Accuracy: 1.0000
Batch 3/7446 (0.0%) | Loss: 1.0960 | Accuracy: 1.0000
Batch 4/7446 (0.1%) | Loss: 1.0960 | Accuracy: 1.0000
Batch 5/7446 (0.1%) | Loss: 1.0960 | Accuracy: 1.0000
Batch 6/7446 (0.1%) | Loss: 1.0960 | Accuracy: 1.0000
Batch 7/7446 (0.1%) | Loss: 1.0960 | Accuracy: 1.0000
Batch 8/7446 (0.1%) | Loss: 1.0960 | Accuracy: 1.0000
Batch 9/7446 (0.1%) | Loss: 1.0960 | Accuracy: 1.0000
Batch 10/7446 (0.1%) | Loss: 1.0960 | Accuracy: 1.0000
Batch 11/7446 (0.1%) | Loss: 1.0960 | Accuracy: 1.0000
Batch 12/7446 (0.2%) | Loss: 1.0960 | Accuracy: 1.0000
Batch 13/7446 (0.2%) | Loss: 1.0960 | Accuracy: 1.0000
Batch 14/7446 (0.2%) | Loss: 1.0960 | Accuracy: 1.0000
Batch 15/7446 (0.2%) | Loss: 1.0960 | Accuracy: 1.0000
Batch 16/7446 (0.2%) | Loss: 1.0960 | Accuracy: 1.0000
Batch 17/7446 (0.2%) | Loss: 1.0960 | Accuracy: 1.0000
Batch 18/7446 (0.2%) | Loss: 1.0960 | Accuracy: 1.0000
Batch 19/7446 (0.3%

In [82]:
from sklearn.metrics import confusion_matrix

all_preds = []
all_labels = []

with torch.no_grad():
    for i, batch in enumerate(dataloader):
        if i >= 200:
            break
        x, y = batch
        x = reshape_batch(x)
        washout_list = torch.tensor([int(washout_rate * x.size(0))] * x.size(1)).to(device)
        x = x.to(device)
        y = y.to(device)

        # predykcja po wytrenowanym readoucie
        y_pred, _ = model(x, washout_list)

        # zamiana one-hot na indeksy klas
        y_index = y.argmax(dim=-1)
        pred_classes = y_pred.argmax(dim=-1)

        # spłaszczenie batch + sekwencje w jeden wymiar
        all_preds.append(pred_classes.reshape(-1).cpu())
        all_labels.append(y_index.reshape(-1).cpu())
        percent_done = 100 * (i + 1) / total_batches
        print(f"Batch {i+1}/{total_batches} ({percent_done:.1f}%)")

# scal wszystkie batchy w jeden tensor
all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)
cm = confusion_matrix(all_labels.numpy(), all_preds.numpy())

Batch 1/7446 (0.0%)
Batch 2/7446 (0.0%)
Batch 3/7446 (0.0%)
Batch 4/7446 (0.1%)
Batch 5/7446 (0.1%)
Batch 6/7446 (0.1%)
Batch 7/7446 (0.1%)
Batch 8/7446 (0.1%)
Batch 9/7446 (0.1%)
Batch 10/7446 (0.1%)
Batch 11/7446 (0.1%)
Batch 12/7446 (0.2%)
Batch 13/7446 (0.2%)
Batch 14/7446 (0.2%)
Batch 15/7446 (0.2%)
Batch 16/7446 (0.2%)
Batch 17/7446 (0.2%)
Batch 18/7446 (0.2%)
Batch 19/7446 (0.3%)
Batch 20/7446 (0.3%)
Batch 21/7446 (0.3%)
Batch 22/7446 (0.3%)
Batch 23/7446 (0.3%)
Batch 24/7446 (0.3%)
Batch 25/7446 (0.3%)
Batch 26/7446 (0.3%)
Batch 27/7446 (0.4%)
Batch 28/7446 (0.4%)
Batch 29/7446 (0.4%)
Batch 30/7446 (0.4%)
Batch 31/7446 (0.4%)
Batch 32/7446 (0.4%)
Batch 33/7446 (0.4%)
Batch 34/7446 (0.5%)
Batch 35/7446 (0.5%)
Batch 36/7446 (0.5%)
Batch 37/7446 (0.5%)
Batch 38/7446 (0.5%)
Batch 39/7446 (0.5%)
Batch 40/7446 (0.5%)
Batch 41/7446 (0.6%)
Batch 42/7446 (0.6%)
Batch 43/7446 (0.6%)
Batch 44/7446 (0.6%)
Batch 45/7446 (0.6%)
Batch 46/7446 (0.6%)
Batch 47/7446 (0.6%)
Batch 48/7446 (0.6%)
B

In [83]:
batch = next(iter(dataloader))

x, y = batch
x = reshape_batch(x)
washout_list = torch.tensor([int(washout_rate * x.size(0))] * x.size(1)).to(device)
x = x.to(device)
y = y.to(device)

# predykcja po wytrenowanym readoucie
y_pred, _ = model(x, washout_list)

y_pred

tensor([[[0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.0039, 0.0000],
         [0.0000, 0.

In [85]:
cm

array([[      0,  400000,       0],
       [      0, 1105940,       0],
       [      0,  400000,       0]])